In [2]:
import os, subprocess, sys

# ── 1. Install uv ────────────────────────────────────────────────────────────
subprocess.run("wget -qO- https://astral.sh/uv/install.sh | sh", shell=True)
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")

# ── 2. Create venv ──────────────────────────────────────────────────────────
subprocess.run("uv venv .venv --seed --clear --python 3.11", shell=True)
pip = ".venv/bin/python -m pip"

# ── 3. Phase 1: Blackwell PyTorch (The Critical Fix) ─────────────────────────
# We use CUDA 12.8 nightly because it contains the sm_120 kernels.
print("Installing Blackwell-ready PyTorch (CUDA 12.8 nightly)...")
subprocess.run(f"""
{pip} install --pre torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/nightly/cu128 --force-reinstall
""", shell=True, check=True)

# ── 4. Phase 2: Math Reasoning Stack ─────────────────────────────────────────
# We install these without version pins for the most part to allow the
# resolver to find Blackwell-compatible wheels (Qwen-2507 needs newer libs).
print("Installing remaining math-reasoning stack...")
subprocess.run(f"""
{pip} install \
    "numpy<2" \
    "transformers>=4.51.0" \
    "accelerate>=0.30" \
    "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "vllm>=0.7.0" \
    "triton" \
    "nvidia-cuda-runtime-cu12" \
    "nvidia-cuda-cupti-cu12" \
    "nvidia-cudnn-cu12" \
    "nvidia-nvjitlink-cu12" \
    "pandas" \
    "antlr4-python3-runtime==4.11.1" \
    sympy tqdm sentencepiece ipykernel jupyter
""", shell=True, check=True)

# ── 5. Environment & Kernel Setup ───────────────────────────────────────────
venv_site = ".venv/lib/python3.11/site-packages"
ld_paths = [
    f"{venv_site}/nvidia/nvjitlink/lib",
    f"{venv_site}/nvidia/cuda_runtime/lib",
    f"{venv_site}/nvidia/cudnn/lib",
    "/usr/local/cuda/lib64",
    "/usr/local/cuda-12.8/lib64",
]
os.environ["LD_LIBRARY_PATH"] = ":".join(ld_paths) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Permanently fix bashrc for background tasks and future sessions
bashrc_line = f'\nexport LD_LIBRARY_PATH={":".join(ld_paths)}:$LD_LIBRARY_PATH\n'
with open(os.path.expanduser("~/.bashrc"), "a") as f:
    f.write(bashrc_line)

subprocess.run(
    ".venv/bin/python -m ipykernel install --user --name cse151bV2 --display-name 'Python (cse151bV2)'",
    shell=True
)
print("\n✅ Environment built for Blackwell (sm_120).")
print("👉 ACTION: Select kernel 'cse151bV2' in the top right → RESTART KERNEL.")

downloading uv 0.11.17 x86_64-unknown-linux-gnu


installing to /home/ubuntu/.local/bin
  uv
  uvx
everything's installed!


Using CPython 3.11.15
Creating virtual environment with seed packages at: .venv
 + packaging==26.2
 + pip==26.1.1
 + setuptools==82.0.1
 + wheel==0.47.0
Activate with: source .venv/bin/activate


Installing Blackwell-ready PyTorch (CUDA 12.8 nightly)...
Looking in indexes: https://download.pytorch.org/whl/nightly/cu128
  Using cached torch-2.12.0.dev20260408%2Bcu128-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.27.0.dev20260407%2Bcu128-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchaudio-2.11.0.dev20260407%2Bcu128-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_bindings-12.9.4-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_

In [7]:
!.venv/bin/pip install torch transformers peft accelerate bitsandbytes datasets trl vllm sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 16.8 MB/s  0:00:00


In [1]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
free, total = torch.cuda.mem_get_info()
print(f"Free: {free/1e9:.1f} GB / {total/1e9:.1f} GB")

Free: 84.2 GB / 85.1 GB


In [3]:
!.venv/bin/pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 12.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 83.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [sentence-transformers]ence-transformers]


In [4]:
!python select_512.py \
    --data-path data/public.jsonl \
    --output-path data/selected_grpo.jsonl \
    --n-samples 128 \
    --n-clusters 16

Loading data from data/public.jsonl ...
  Loaded 1126 items
  MCQ: 375  |  Free-form: 751
Computing SBERT embeddings (all-MiniLM-L6-v2) ...
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 345kB/s]
README.md: 10.5kB [00:00, 23.4MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 246kB/s]
config.json: 100%|█████████████████████████████| 612/612 [00:00<00:00, 1.92MB/s]
model.safetensors: 100%|███████████████████| 90.9M/90.9M [00:01<00:00, 73.0MB/s]
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 1.81MB/s]
vocab.txt: 232kB [00:00, 14.3MB/s]
tokenizer.json: 466kB [00:00, 26.9MB/s]
Batches: 100%|██████████████████████████████████| 18/18 [00:03<00:00,  5.65it/s]
  Embeddings shape: (1126, 384)
Running KMeans with k=16 ...

Allocation summary (total=128):
  Cluster  0: size=  75  →  pick=8
  Cluster  1: size=  73  →  pick=8
  Cluster  2: size=  95  →  pick=11
  Cluster  3: size=  87  →  pick=10
  Cluster  4: size=  40  → 

In [ ]:
!.venv/bin/python -u train_grpo_qwen_responses_nothink.py \
    --init-adapter data/lora_math_adaper/lora_math_adapter/final_adapter \
    --max-completion 10000 \
    --vllm-mem 0.55 \
    --epochs 1 \
    --grad-accum 4 \
    --num-generations 4 \
    --save-every-steps 10 \
    --data data/selected_grpo.jsonl


[23:40:01] STEP 1 / 6  CUDA sanity check (bf16, L40 48GB profile)
  [23:40:05] torch 2.9.1+cu128 cuda 12.8
  [23:40:05] device: NVIDIA A100-SXM4-80GB
  [23:40:05] compute capability 8.0
  [23:40:05] note: cc 8.0 (Ampere/Ada) — bf16 path OK
  [23:40:05] training precision: bf16 + tf32

[23:40:05] STEP 2 / 6  Imports
  [23:40:05] importing ...
  [23:40:21] imports done.
  [23:40:21] judger + reward functions ready (correctness + format + completion).

[23:40:21] STEP 3 / 6  Load tokenizer + 4-bit base model + LoRA
  [23:40:22] flash-attn not installed — using sdpa
  [23:40:22] attention backend: sdpa
Loading checkpoint shards: 100%|██████████████████| 3/3 [01:01<00:00, 20.57s/it]
  [23:41:25] base model loaded in 62.9s.
  [23:41:25] loading LoRA from data/lora_math_adaper/lora_math_adapter/final_adapter ...
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

[23:42:13] STEP 4 / 6  Build prompt dataset
  [23:42:13] loaded 128 problems.
  [23:42:13] prompt tok